# Data Generation: ARES Beamline Simulation

This notebook generates a dataset for training a surrogate model of the ARES 
accelerator beamline at DESY. We use Cheetah, a differentiable beam dynamics 
simulator, to track electron beams through a sequence of quadrupole magnets 
and correctors.

The goal: learn the mapping from magnet settings (inputs) to beam properties 
at the screen (outputs), which is exactly what a surrogate model does.

In [18]:
import cheetah
import torch
import numpy as np
import pandas as pd

## 1. Beamline Definition

The ARES beamline consists of 3 quadrupole magnets (AREAMQZM1-3), 
a vertical corrector (AREAMCVM1), a horizontal corrector (AREAMCHM1), 
and a screen (AREABSCR1) where the beam properties are measured.

- **Inputs (control parameters):** k1 of each quadrupole, angle of each corrector
- **Outputs (beam properties):** sigma_x, sigma_y, mu_x, mu_y at the screen

In [19]:
segment = cheetah.Segment(
    elements=[
        cheetah.Drift(length=torch.tensor(0.175)),
        cheetah.Quadrupole(length=torch.tensor(0.122), name="AREAMQZM1"),
        cheetah.Drift(length=torch.tensor(0.428)),
        cheetah.Quadrupole(length=torch.tensor(0.122), name="AREAMQZM2"),
        cheetah.Drift(length=torch.tensor(0.204)),
        cheetah.VerticalCorrector(length=torch.tensor(0.02), name="AREAMCVM1"),
        cheetah.Drift(length=torch.tensor(0.204)),
        cheetah.Quadrupole(length=torch.tensor(0.122), name="AREAMQZM3"),
        cheetah.Drift(length=torch.tensor(0.179)),
        cheetah.HorizontalCorrector(length=torch.tensor(0.02), name="AREAMCHM1"),
        cheetah.Drift(length=torch.tensor(0.45)),
        cheetah.Screen(name="AREABSCR1"),
    ]
)

parameter_beam = cheetah.ParameterBeam.from_twiss(beta_x=torch.tensor(3.14))

## 2. Dataset Generation

We sample 10,000 random configurations of the 5 control parameters and simulate 
the resulting beam properties using Cheetah.

Parameter ranges are taken from the official DESY rl-vs-bo repository, 
reflecting the real operating limits of the ARES accelerator:

- Quadrupole strengths (k1): [-72, 72] m⁻²
- Corrector angles: [-6.1782e-3, 6.1782e-3] rad

In [20]:
N = 10000
records = []

for i in range(N):
    k1_q1 = np.random.uniform(-72, 72)
    k1_q2 = np.random.uniform(-72, 72)
    k1_q3 = np.random.uniform(-72, 72)
    angle_v = np.random.uniform(-6.1782e-3, 6.1782e-3)
    angle_h = np.random.uniform(-6.1782e-3, 6.1782e-3)    
    
    segment.AREAMQZM1.k1 = torch.tensor(k1_q1)
    segment.AREAMQZM2.k1 = torch.tensor(k1_q2)
    segment.AREAMQZM3.k1 = torch.tensor(k1_q3)
    segment.AREAMCVM1.angle = torch.tensor(angle_v)
    segment.AREAMCHM1.angle = torch.tensor(angle_h)
    
    out = segment.track(parameter_beam)
    
    records.append({
        "k1_q1": k1_q1,
        "k1_q2": k1_q2,
        "k1_q3": k1_q3,
        "angle_v": angle_v,
        "angle_h": angle_h,
        "sigma_x": out.sigma_x.item(),
        "sigma_y" : out.sigma_y.item(),
        "mu_x": out.mu_x.item(),
        "mu_y": out.mu_y.item()
        
    })
    
df = pd.DataFrame(records)
print(df.head())    

       k1_q1      k1_q2      k1_q3   angle_v   angle_h   sigma_x  \
0 -57.642280 -30.893184   8.168233  0.005502 -0.002878  0.000042   
1 -10.735566  62.153794 -54.984831 -0.003938  0.000951  0.000047   
2  59.930030 -17.791167 -24.374194  0.005539  0.001757  0.000056   
3 -49.247187 -44.732797 -47.618584  0.004088 -0.003265  0.000240   
4  69.408039 -15.492915 -47.081076  0.000009 -0.005843  0.000097   

        sigma_y      mu_x          mu_y  
0  6.794602e-07 -0.001295  6.406574e-03  
1  7.302314e-06  0.000428  5.462379e-04  
2  5.441361e-06  0.000791  2.477597e-03  
3  1.671148e-06 -0.001469 -2.131943e-05  
4  1.403287e-05 -0.002629  4.267724e-08  


In [21]:
df.to_csv('../data/beam_dataset.csv', index=False)
print(f"Dataset saved: {df.shape}")
print(df.describe())

Dataset saved: (10000, 9)
              k1_q1         k1_q2         k1_q3       angle_v       angle_h  \
count  10000.000000  10000.000000  10000.000000  10000.000000  10000.000000   
mean       0.143491     -0.669667     -0.248683      0.000069      0.000004   
std       41.314820     41.623367     41.500595      0.003595      0.003573   
min      -71.998199    -71.999431    -71.987945     -0.006177     -0.006177   
25%      -35.825480    -36.701827    -35.639847     -0.003045     -0.003082   
50%        0.009727     -1.126797     -0.683080      0.000100      0.000019   
75%       35.446443     35.475269     35.725436      0.003204      0.003100   
max       71.991313     71.983093     71.996599      0.006178      0.006178   

            sigma_x       sigma_y          mu_x          mu_y  
count  1.000000e+04  1.000000e+04  10000.000000  10000.000000  
mean   3.896263e-05  2.313037e-05      0.000002      0.000038  
std    5.969543e-05  3.534599e-05      0.001608      0.005050  
min   